# FphIShE: testing the MVP follow-ups

**Open in Colab:** https://colab.research.google.com/github/Eitzl/FphIShE/blob/claude/jolly-goldberg-f80xci/notebooks/test_improvements.ipynb

The MVP run (2026-10-30) left two questions. This notebook answers them on real data:

1. **Where does the prefilter lose homologs?** Sensitivity stayed at about 0.25 for every k from 25
   to 400, while full Smith-Waterman got 0.293. The suspects are the IVF index (`nprobe`), the PCA
   reduction (`dim`), the distance (`metric`), and the embedding itself.
   `diagnose_prefilter.py` blames each lost homolog on one of these stages.
2. **What does fphishe cost end to end?** The report now adds query embedding time to the search
   time, records the hardware, and reports medians of 3 timing runs.

It also splits sensitivity into close homologs (`family`) and remote ones (`remote`).

Run top to bottom on any GPU runtime. Total: about 20 min with saved embeddings.
At the end, send back the output of the last two cells.

## 0. Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!nproc

In [ ]:
!git clone -b claude/jolly-goldberg-f80xci https://github.com/Eitzl/FphIShE.git /content/FphIShE
%cd /content/FphIShE
!git log --oneline -3

If the repo is private, the clone above fails. Use the token cells from `run_mvp.ipynb` (section 0.2) instead, with this branch.

In [ ]:
!pip install -q -e ".[align,gpu,embed,dev]"
!apt-get -qq install -y ncbi-blast+ > /dev/null
!pytest -q

## 1. Data and embeddings

Restores `embeddings.h5` from the MVP run. If there's no saved copy, the second cell embeds from scratch (about 2 min on an A100, 10–30 min on a T4).

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/fphishe data/processed
!python scripts/download_data.py
!cp -v /content/drive/MyDrive/fphishe/embeddings.h5 data/processed/ 2>/dev/null || echo "no saved embeddings"

In [ ]:
!python scripts/embed.py
!python -c "import h5py; print(len(h5py.File('data/processed/embeddings.h5','r')), 'embedded (expect 15177)')"
!cp -n data/processed/embeddings.h5 /content/drive/MyDrive/fphishe/

## 2. Queries and query embedding time

Same seed as the MVP run, so these are the same 500 queries. The second line starts a clean
`timings.tsv`. The last cell embeds the 500 queries again into a separate file to measure what a
new query costs (logged as `embed`).

In [ ]:
!python scripts/pick_queries.py
!rm -f results/timings.tsv results/hits_*.tsv results/summary_*.tsv

In [ ]:
!rm -f data/processed/query_embeddings.h5
!python scripts/embed.py --ids data/processed/queries.txt --out data/processed/query_embeddings.h5

## 3. Baselines

Full Smith-Waterman (CPU) and BLAST, each timed 3 times. `diagnose_prefilter.py` needs the SW hits.

In [ ]:
!python scripts/search_sw.py --repeats 3
!python scripts/search_blast.py --repeats 3

## 4. Prefilter experiments

Each config runs fphishe, the report, and the diagnosis. What to look for in each diagnosis:

- `baseline`: the MVP settings (`--dim 128 --nprobe 10`, L2).
- `exact_ivf`: `--nprobe 100` searches every IVF cluster. "lost to ivf" should drop to 0, and the
  sensitivity gain shows what the approximation costs.
- `dim256` / `dim1024`: less or no PCA compression. "lost to pca" should shrink.
- `cos`: cosine similarity instead of L2.
- `cos_full_exact`: everything exact. This is the ceiling of the embedding alone.

If most lost homologs are blamed on `embedding`, tuning the index won't help and the next step is
the rerank score (fusing embedding similarity with the SW score).

In [ ]:
configs = {
    "baseline": "",
    "exact_ivf": "--nprobe 100",
    "dim256": "--dim 256",
    "dim1024": "--dim 1024",
    "cos": "--metric cos",
    "cos_full_exact": "--metric cos --dim 1024 --nprobe 100",
}
for name, flags in configs.items():
    print(f"=== {name}: {flags or '(defaults)'}")
    !python scripts/search_fphishe.py {flags} --repeats 3
    !python scripts/report.py > results/report_{name}.txt && cp results/summary.tsv results/summary_{name}.tsv
    !python scripts/diagnose_prefilter.py {flags} | tee results/diagnose_{name}.txt

## 5. k sweep with the best config

Set `BEST` to the flags of the config with the highest fphishe sensitivity in section 6 (run
section 6 first if you like). The MVP sweep used the baseline; this shows whether a better
prefilter changes the flat sensitivity curve.

In [ ]:
BEST = "--metric cos"  # edit after looking at section 6
for k in [25, 50, 100, 200, 400]:
    !python scripts/search_fphishe.py {BEST} --k {k} > /dev/null
    !python scripts/report.py --k {k} > /dev/null && cp results/summary.tsv results/summary_k{k}.tsv
    !python scripts/diagnose_prefilter.py {BEST} --k {k} | grep -E "kept|embedding-lost"

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

ks = [25, 50, 100, 200, 400]
sweep = [pd.read_csv(f"results/summary_k{k}.tsv", sep="\t").set_index("method") for k in ks]
plt.plot(ks, [s.loc["fphishe", "sens_mean"] for s in sweep], "o-", label="fphishe sensitivity")
plt.plot(ks, [s.loc["prefilter"].filter(like="recall@").iloc[0] for s in sweep], "s-",
         label="prefilter recall@k")
plt.axhline(sweep[0].loc["sw", "sens_mean"], color="gray", ls="--", label="sw sensitivity")
plt.xscale("log")
plt.xlabel("k")
plt.legend()
plt.savefig("results/k_sweep.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Results to send back

The comparison table (fphishe per config, with sw and blast for reference), then the baseline report and every diagnosis.

In [ ]:
rows = []
base = pd.read_csv("results/summary_baseline.tsv", sep="\t").set_index("method")
for m in ["sw", "blast"]:
    r = base.loc[m]
    rows.append({"config": m, "sens": r.sens_mean, "family": r.sens_family,
                 "remote": r.sens_remote, "recall@100": r["recall@100"],
                 "ms/query": 1000 * r.seconds / r.n_queries, "device": r.device})
for name in configs:
    s = pd.read_csv(f"results/summary_{name}.tsv", sep="\t").set_index("method")
    r = s.loc["fphishe"]
    rows.append({"config": f"fphishe {name}", "sens": r.sens_mean, "family": r.sens_family,
                 "remote": r.sens_remote, "recall@100": r["recall@100"],
                 "p vs sw": s.loc["sw", "p_vs_fphishe"],
                 "ms/query": 1000 * r.seconds / r.n_queries, "device": r.device})
table = pd.DataFrame(rows).set_index("config").round(3)
table.to_csv("results/comparison.tsv", sep="\t")
table

In [ ]:
!cat results/report_baseline.txt
!tail -n +1 results/diagnose_*.txt
!cp -r results /content/drive/MyDrive/fphishe/results_tests_$(date +%Y%m%d_%H%M)